# Concept Population Builder — Barrett-Aligned LLM Vocabulary Workflow

## Theoretical Foundation

This notebook demonstrates a concept-learning workflow grounded in **Lisa Feldman Barrett's
Theory of Constructed Emotion** (*How Emotions Are Made*, 2017).

Barrett argues that concepts are not fixed definitions stored in memory. Instead, the brain
maintains a **population of variable instances** — past predictions clustered by a shared
*functional goal*, not by perceptual similarity. Each instance is indexed by:
- **Context**: the specific situation the concept is grounded in.
- **Goal**: what the concept is supposed to accomplish in that context.
- **Simulation**: the brain's forward prediction of what this concept produces.

**Implication for LLMs:** Current models store token statistics, not goal-indexed
conceptual populations. This workflow introduces exactly that missing layer — building
a `ConceptPopulation` for a given term through iterative RL scoring and human RLHF feedback.

---
**Key vocabulary (Barrett-aligned, used throughout):**
- `simulation` — the predicted experience/response (NOT "definition" or "answer")
- `instance` — one contextual entry in the population (NOT "example")
- `functional adequacy` — how well the simulation serves the goal in context (NOT "accuracy" or "clarity")
- `population` — the full set of instances for a term (NOT "dictionary")

In [ ]:
# Cell 2 — Setup: imports and environment
import os, sys, json
from pathlib import Path

# Ensure the project root is on the path when running from notebooks/
project_root = Path(".").resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env")

stub_mode = os.getenv("WATSONX_STUB", "false").lower() == "true"
print(f"WATSONX_STUB mode: {stub_mode}")
print(f"Model: {os.getenv('WATSONX_MODEL_ID', 'ibm/granite-13b-instruct-v2')}")
print("✅ Environment loaded.")

In [ ]:
# Cell 3 — Define concept term and (context, goal) pairs for "anger"
# These pairs represent the population we want to build.
# Each pair is a distinct (context, goal) frame — the concept behaves differently in each.

TERM = "anger"

CONTEXT_GOAL_PAIRS = [
    (
        "receiving unfair criticism at work from a manager",
        "restore social fairness and protect professional reputation"
    ),
    (
        "watching a news story about injustice toward a vulnerable group",
        "motivate collective action and social change"
    ),
    (
        "a close friend breaking a promise that caused significant inconvenience",
        "signal boundary violation and renegotiate the relationship norm"
    ),
]

print(f"Term: {TERM}")
print(f"Building a population from {len(CONTEXT_GOAL_PAIRS)} (context, goal) pairs:")
for i, (ctx, goal) in enumerate(CONTEXT_GOAL_PAIRS, 1):
    print(f"  {i}. Context: {ctx[:60]}")
    print(f"     Goal:    {goal[:60]}")

In [ ]:
# Cell 4 — Run the RL loop and display interim results
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

from src.concept_loop import run_rl_loop

population = run_rl_loop(
    term=TERM,
    context_goal_pairs=CONTEXT_GOAL_PAIRS,
    max_iterations=3,
    threshold=7.5,
)

print(f"\n{'='*60}")
print(f"RL loop complete.")
print(f"  Population breadth : {population.population_breadth}")
print(f"  Goal coverage      : {len(population.goal_coverage)} distinct goals")
print(f"  Context coverage   : {len(population.context_coverage)} distinct contexts")
print(f"{'='*60}")

# Show each instance with its final round and adequacy score
for i, inst in enumerate(population.instances, 1):
    score_str = f"{inst.adequacy_score:.1f}/10" if inst.adequacy_score is not None else "N/A"
    print(f"\n[Instance {i}]")
    print(f"  Context  : {inst.context}")
    print(f"  Goal     : {inst.goal}")
    print(f"  Simulation (Round {inst.round}):")
    print(f"    {inst.simulation}")
    print(f"  Adequacy : {score_str}")
    if inst.history:
        print(f"  History  : {len(inst.history)} round(s) tracked")

In [ ]:
# Cell 5 — Display the final ConceptPopulation as a formatted table
try:
    from rich.table import Table
    from rich.console import Console
    from rich.text import Text
    _rich = True
except ImportError:
    _rich = False

if _rich:
    console = Console()
    table = Table(title=f"ConceptPopulation: '{TERM}'", show_lines=True)
    table.add_column("#", style="dim", width=3)
    table.add_column("Context", max_width=35)
    table.add_column("Goal", max_width=35)
    table.add_column("Simulation", max_width=50)
    table.add_column("Round", justify="center", width=6)
    table.add_column("Score", justify="center", width=7)
    table.add_column("Human", justify="center", width=8)

    for i, inst in enumerate(population.instances, 1):
        score_str = f"{inst.adequacy_score:.1f}" if inst.adequacy_score is not None else "N/A"
        sig = inst.human_signal or "—"
        table.add_row(
            str(i),
            inst.context[:35],
            inst.goal[:35],
            inst.simulation[:50],
            str(inst.round),
            score_str,
            sig,
        )
    console.print(table)
else:
    # Fallback: plain print
    print(f"{'#':>2}  {'Context':30}  {'Score':6}  Simulation")
    print("-" * 80)
    for i, inst in enumerate(population.instances, 1):
        score_str = f"{inst.adequacy_score:.1f}" if inst.adequacy_score is not None else "N/A"
        print(f"{i:>2}  {inst.context[:30]:30}  {score_str:6}  {inst.simulation[:50]}")

In [ ]:
# Cell 6 — Generate the Concept Population Report
import os
from pathlib import Path
from src.report import generate_report

report_output = str(project_root / "docs" / "concept_population_report.md")
result = generate_report(population, output_path=report_output)

print(f"\nReport path : {result['report_path']}")
print(f"Metrics     : {json.dumps(result['metrics'], indent=2)}")

# Also save the ConceptPopulation JSON
json_path = project_root / "docs" / f"{TERM}_population.json"
(project_root / "docs").mkdir(parents=True, exist_ok=True)
with open(json_path, "w", encoding="utf-8") as fh:
    fh.write(population.to_json())
print(f"Population JSON saved to: {json_path}")

## Concluding Notes

### What we built

This notebook ran the **Barrett-aligned concept population workflow** end-to-end:

1. **Population construction** — for each (context, goal) pair, `run_rl_loop` generated
   an initial simulation using watsonx.ai's generative model.

2. **Functional adequacy scoring** — each simulation was scored by a judge LLM prompt
   asking: *"How well does this simulation predict what the concept produces in this context
   to serve this goal?"*

3. **RL refinement** — instances below the adequacy threshold were automatically refined
   and re-scored, with full history tracked per instance for score-delta reporting.

4. **Concept Population Report** — metrics, per-instance table, and a population verdict
   written to `docs/concept_population_report.md`.

### Barrett alignment

Every element of this workflow mirrors Barrett's constructionist model:
- The population **grows** — instances are added, never replaced.
- Each instance is **goal-indexed** — the same term produces different simulations for different goals.
- Refinement is driven by **prediction error** — the adequacy score is the model's signal
  that the current simulation failed to predict well enough.
- Human RLHF mirrors **social reality feedback** — the external signal that updates the population.

### Next steps

- Set `WATSONX_STUB=false` in `.env` and supply real credentials for live API runs.
- Run `collect_human_feedback(population, term=TERM)` to add the interactive RLHF layer.
- Expand `CONTEXT_GOAL_PAIRS` to test population breadth across more diverse contexts.
- Compare population quality across different `WATSONX_MODEL_ID` values.